[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_30/1_algorithms.ipynb)

# Day 30: Algorithms (final check)

**Light day.** This is a short final check, not a new exam: about 45 minutes, no stress. Solve the two
problems with the five-step script from yesterday (clarify, examples, brute force, optimise, test), then do the
10-minute pattern drill out loud. Finish with the checklist at the end and stop studying algorithms for today.

**How to work:** Name the pattern before you code. Stuck 15 min: open Hint 1. Stuck 25 min: Hint 2, then the solution. Say the time and space complexity out loud. Re-type the solution from memory if you needed it.

**Today (45 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Convoys on a one-lane road | medium | 15 |
| Q2 | Top creators by watch time from raw player events | medium | 20 |
| Q3 | Rapid drill: name the method (review) | mixed | 10 |

**Handbook:** [Algorithms](https://mahsa-agz.github.io/ds-handbook/#algo), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import copy

def check(fn, cases, key=None):
    """Runs fn on each (args, expected) case and prints PASS or FAIL. key: normalise outputs (e.g. sorted)."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            got = "error: " + repr(e)
        good = (key(got) == key(want)) if (key and not isinstance(got, str)) else got == want
        ok += good
        shown = ", ".join(repr(a) for a in args)
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}({shown}) -> {got!r}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

import random, signal, time

class _TooSlow(BaseException):
    pass

def _stop(signum, frame):
    raise _TooSlow()

def check_big(label, run, want, key=None, limit=1.0):
    """Big-input test (the input is not printed). run() calls your function.
    PASS only if the answer is right AND it finishes within `limit` seconds.
    In Colab (Linux) a run that takes longer than 3 * limit + 3 seconds is stopped, so a slow brute force cannot hang the notebook."""
    alarm = hasattr(signal, "setitimer")
    if alarm:
        old = signal.signal(signal.SIGALRM, _stop)
        signal.setitimer(signal.ITIMER_REAL, 3 * limit + 3)
    t = time.perf_counter()
    try:
        got = run()
        if key is not None:
            got = key(got)
    except _TooSlow:
        got = "stopped: too slow"
    except Exception as e:
        got = "error: " + repr(e)[:120]
    finally:
        if alarm:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old)
    sec = time.perf_counter() - t
    good = got == want
    fast = sec <= limit
    note = ("" if good else "  wrong answer" + (f" ({got})" if isinstance(got, str) else "")) + ("" if fast else "  too slow")
    print(("PASS " if good and fast else "FAIL ") + f"{label}: {sec:.2f}s (limit {limit}s)" + note)

def make_watch_log(seed=30):
    """MADE-UP raw player events of a short-video app.
    watch_events: rows (ts_seconds, user_id, video_id, event) with event in "play", "pause", "stop".
    videos: catalog video_id -> creator_id (a few deleted videos are missing from it).
    Like real logs: unsorted, retried duplicates, pauses with no play before them, double plays,
    and views that never stop (the app crashed)."""
    rng = random.Random(seed)
    creators = [f"c{i:02d}" for i in range(1, 41)]
    pop = {c: rng.paretovariate(1.2) for c in creators}                 # a few creators are much more popular
    all_videos = {f"v{i:04d}": rng.choice(creators) for i in range(1, 401)}
    ids = sorted(all_videos)
    weights = [pop[all_videos[v]] for v in ids]
    rows = []
    for u in range(1, 601):
        user = f"u{u:04d}"
        for video in set(rng.choices(ids, weights=weights, k=rng.randint(1, 12))):
            t = rng.randint(0, 86000)
            if rng.random() < 0.02:
                rows.append((t - rng.randint(1, 100), user, video, "pause"))      # pause with no play before it
            for seg in range(rng.randint(1, 4)):                                   # play ... pause segments
                rows.append((t, user, video, "play"))
                dur = rng.randint(2, 120)
                if rng.random() < 0.02:
                    rows.append((t + 1, user, video, "play"))                      # double play: ignored
                t += dur
                rows.append((t, user, video, "pause"))
                t += rng.randint(1, 300)
            if rng.random() < 0.9:
                rows[-1] = (rows[-1][0], user, video, "stop")                      # last pause becomes a stop
            else:
                rows.append((t, user, video, "play"))                              # never closed: app crashed
    rows += rng.sample(rows, len(rows) // 30)                                      # retried duplicates
    rng.shuffle(rows)
    deleted = set(rng.sample(ids, 10))
    videos = {v: c for v, c in all_videos.items() if v not in deleted}
    return rows, videos

watch_events, videos = make_watch_log()
print(len(watch_events), "events,", len(videos), "videos in the catalog, for example:", watch_events[:2])

---
### Q1. Convoys on a one-lane road

*medium, about 15 min*

Delivery trucks drive on a one-lane road towards a depot at mile `target`. Truck `i` starts at mile
`position[i]` (all different) and drives at `speed[i]` miles per hour. A truck can never pass another one: when it
catches up with a slower truck, it slows down and they drive on together as one **convoy** (a convoy can grow).
A truck that catches up exactly at the depot also counts as part of that convoy. Return the number of convoys that
arrive at the depot.

Examples:
- `target = 20, position = [0, 4, 10, 16], speed = [4, 2, 2, 1]` -> `3`. The truck at 16 needs 4 hours, the one at
  10 needs 5 hours (cannot catch it), the one at 4 needs 8 hours, and the one at 0 would need 5 hours, so it catches
  the truck from 4 and joins it.
- `target = 10, position = [0, 5], speed = [2, 1]` -> `1` (both would arrive after 5 hours: they meet at the depot)

Constraints: `0 <= n <= 10**5`. Target: O(n log n). The big test has 100000 trucks.

*Source: [LeetCode 853](https://leetcode.com/problems/car-fleet/)*

In [ ]:
def car_fleet(target, position, speed):
    pass

# tests
check(car_fleet, [
    ((20, [0, 4, 10, 16], [4, 2, 2, 1]), 3),
    ((10, [3], [3]), 1),
    ((10, [], []), 0),
    ((100, [0, 2, 4], [4, 2, 1]), 1),
    ((10, [6, 8], [3, 2]), 2),
    ((10, [0, 4, 8], [1, 1, 1]), 3),
    ((12, [4, 6], [6, 3]), 1),
    ((10, [0, 5], [2, 1]), 1),
])
# big test: 100000 trucks
rng = random.Random(8530)
big_pos = rng.sample(range(10**6), 100000)
big_speed = [rng.randint(1, 10**3) for _ in range(100000)]
check_big("n = 100000", lambda: car_fleet(10**6, big_pos, big_speed), 132, limit=1.0)

<details><summary><b>Hint 1</b></summary>

Signal: objects that cannot pass each other, so only the order by position matters, and each one is compared with the group in front of it. Pattern: **sort, then one pass** (a monotonic stack of arrival times also works).

</details>

<details><summary><b>Hint 2</b></summary>

1. Arrival time if alone: `t = (target - position) / speed`.
2. Sort trucks by position, **closest to the depot first**.
3. Keep `lead_time`, the arrival time of the convoy just in front. If `t > lead_time`, this truck cannot catch it:
   new convoy, `lead_time = t`. Otherwise it joins (and is slowed down to `lead_time`).
4. The answer is the number of new convoys.

</details>

<details><summary><b>Solution</b></summary>

```python
def car_fleet_brute(target, position, speed):
    """A truck leads its own convoy only if no truck ahead of it arrives later or at the same time. O(n^2)."""
    times = [(target - p) / s for p, s in zip(position, speed)]
    fleets = 0
    for i in range(len(position)):
        blocked = any(position[j] > position[i] and times[j] >= times[i] for j in range(len(position)))
        fleets += not blocked
    return fleets

def car_fleet(target, position, speed):
    """Sort by position, closest to the target first. Keep the arrival time of the convoy in front:
    a slower-or-equal truck behind joins it, a truck that would arrive later starts a new convoy. O(n log n)."""
    fleets, lead_time = 0, 0.0
    for p, s in sorted(zip(position, speed), reverse=True):
        t = (target - p) / s                  # arrival time if the road were free
        if t > lead_time:                     # cannot catch the convoy ahead: new convoy
            fleets += 1
            lead_time = t
    return fleets

rng = random.Random(853)
for _ in range(3000):
    n = rng.randint(0, 8)
    pos = rng.sample(range(0, 30), n)
    spd = [rng.randint(1, 5) for _ in range(n)]
    assert car_fleet(30, pos, spd) == car_fleet_brute(30, pos, spd), (pos, spd)
print("stress test vs brute force: OK")   # prints: stress test vs brute force: OK
```

**Why:** A truck behind can only be slowed down by the convoy directly in front of it, and once it joins, the
convoy arrives at the time of its leader. So after sorting by position from the front, every truck either starts a
new convoy (it would arrive later than the convoy ahead) or merges (it would arrive earlier or at the same time).
Sorting dominates the cost. The brute force asks, for every truck, whether any truck ahead arrives later, which is
O(n^2).

**Follow-ups the examiner may ask:**
- Floating point: compare `(target - p1) * s2` with `(target - p2) * s1` to stay exact with integers.
- When does each truck reach the depot (LeetCode 1776, Car Fleet II, asks for the collision times)? Monotonic stack
  from the front.
- Positions are already sorted: O(n).
- Streaming: trucks join at the back over time: keep the last `lead_time`.

**Complexity:** Brute force: O(n^2). Sort and scan: O(n log n) time, O(n) space for the sorted pairs.

**Common mistakes:** Sorting from the back of the road instead of the front. Using `>=` instead of `>` (a truck that ties at the depot must join). Integer division for the time. Forgetting the empty input.

**Learn more:** [algo-sorting-intervals](https://mahsa-agz.github.io/ds-handbook/#algo-sorting-intervals), [algo-monotonic-stack](https://mahsa-agz.github.io/ds-handbook/#algo-monotonic-stack), [algo-greedy](https://mahsa-agz.github.io/ds-handbook/#algo-greedy)

</details>

---
### Q2. Top creators by watch time from raw player events

*medium, about 20 min*

The setup cell created two **made-up** objects:
- `watch_events`: about 20000 raw player events `(ts, user_id, video_id, event)`, where `ts` is in seconds and
  `event` is `"play"`, `"pause"` or `"stop"`. The log is unsorted and messy, like a real one.
- `videos`: the catalog `video_id -> creator_id` (a few deleted videos are missing from it).

Write `top_creators(events, videos, k)` that returns the `k` creators with the most **watch time**, as a list of
`(creator_id, seconds)`, sorted by seconds (largest first), ties by `creator_id` (smallest first). Only creators with
more than 0 seconds appear. Watch time is computed per `(user_id, video_id)` pair, with the events in time order:
1. `"play"` opens a segment if none is open (a second `"play"` while one is open is ignored).
2. `"pause"` or `"stop"` closes the open segment and adds `close_ts - open_ts` seconds (ignored if nothing is open).
3. A segment still open at the end of the log is dropped (we do not know when it ended).
4. Events of videos that are not in `videos` are ignored. Exact duplicate rows must not change the result (with
   these rules they do not).

Example with `videos = {"v1": "alice", "v2": "bob"}`:
```
(0,  "u1", "v1", "play"), (100, "u1", "v1", "stop")
(10, "u2", "v2", "play"), (70,  "u2", "v2", "pause"), (80, "u2", "v2", "play"), (90, "u2", "v2", "stop")
-> top 2: [("alice", 100), ("bob", 70)]       (bob: 60 + 10 seconds)
```
Ask first in a real exam: can events arrive out of order (yes), what about duplicates and crashes (rules 1 to
3), and what to do on ties (rule above).

In [ ]:
def top_creators(events, videos, k):
    pass

# tests
V = {"v1": "alice", "v2": "bob", "v3": "alice", "v4": "cara"}
basic = [(0, "u1", "v1", "play"), (100, "u1", "v1", "stop"),
         (10, "u2", "v2", "play"), (70, "u2", "v2", "pause"), (80, "u2", "v2", "play"), (90, "u2", "v2", "stop")]
check(top_creators, [
    # play/stop and play/pause/play/stop segments
    ((basic, V, 2), [("alice", 100), ("bob", 70)]),
    # the same events in another order
    ((basic[::-1], V, 2), [("alice", 100), ("bob", 70)]),
    # equal totals: smaller creator id first, and k cuts the list
    (([(0, "u1", "v1", "play"), (50, "u1", "v1", "stop"), (0, "u1", "v2", "play"), (50, "u1", "v2", "stop")], V, 1),
     [("alice", 50)]),
    # pause with nothing open, double play, and a view on v3 that never stops
    (([(5, "u1", "v1", "pause"), (10, "u1", "v1", "play"), (20, "u1", "v1", "play"), (40, "u1", "v1", "pause"),
       (50, "u1", "v3", "play")], V, 3), [("alice", 30)]),
    # a video missing from the catalog is ignored
    (([(0, "u1", "v9", "play"), (500, "u1", "v9", "stop"), (0, "u1", "v4", "play"), (5, "u1", "v4", "stop")], V, 3),
     [("cara", 5)]),
    # two users on the same video at the same time
    (([(0, "u1", "v2", "play"), (5, "u2", "v2", "play"), (10, "u1", "v2", "stop"), (25, "u2", "v2", "stop")], V, 3),
     [("bob", 30)]),
    # retried duplicate rows count once
    (([(0, "u1", "v1", "play"), (0, "u1", "v1", "play"), (30, "u1", "v1", "stop"), (30, "u1", "v1", "stop")], V, 3),
     [("alice", 30)]),
    # empty log
    (([], V, 3), []),
])
# big test: the made-up log from the setup cell, top 5
check_big(f"watch_events ({len(watch_events)} rows), k = 5", lambda: top_creators(watch_events, videos, 5),
          [('c12', 163529), ('c11', 129188), ('c09', 69323), ('c19', 33531), ('c34', 12081)], limit=1.0)

<details><summary><b>Hint 1</b></summary>

Signal: raw events must be turned into durations per entity, then aggregated and ranked. Pattern: **data processing: group by key, sort each group, a small state machine, then top k with a heap**.

</details>

<details><summary><b>Hint 2</b></summary>

1. Group: `pairs[(user, video)].append((ts, event))`, skipping videos not in the catalog.
2. For each pair, sort by time and walk it with one variable `open_at` (None when nothing is open).
3. Add each pair's seconds to `per_creator[videos[video]]`.
4. Top k: `heapq.nsmallest(k, items, key=lambda x: (-seconds, creator))`, or sort if the number of creators is small.

</details>

<details><summary><b>Solution</b></summary>

```python
import heapq
from collections import defaultdict

def _watch_seconds(events_of_pair):
    """Sum of closed play segments of one (user, video) pair. Events must be in time order."""
    total, open_at = 0, None
    for ts, ev in events_of_pair:
        if ev == "play":
            if open_at is None:                   # a second play while open is ignored
                open_at = ts
        elif open_at is not None:                 # pause or stop closes the open segment
            total += ts - open_at
            open_at = None
    return total                                  # a segment still open at the end is dropped

def top_creators_brute(events, videos, k):
    """For every creator, scan the whole log again for that creator's events. O(C * n log n)."""
    totals = {}
    for creator in set(videos.values()):
        pairs = defaultdict(list)
        for ts, user, video, ev in events:
            if videos.get(video) == creator:
                pairs[(user, video)].append((ts, ev))
        secs = sum(_watch_seconds(sorted(evs)) for evs in pairs.values())
        if secs > 0:
            totals[creator] = secs
    return sorted(totals.items(), key=lambda kv: (-kv[1], kv[0]))[:k]

def top_creators(events, videos, k):
    """One pass to group by (user, video), sort each group, sum per creator, then a heap for the top k.
    O(n log n) for the sorts + O(C log k) for the top k."""
    pairs = defaultdict(list)
    for ts, user, video, ev in events:
        if video in videos:                        # deleted / unknown videos are ignored
            pairs[(user, video)].append((ts, ev))
    per_creator = defaultdict(int)
    for (user, video), evs in pairs.items():
        evs.sort()                                 # time order; exact duplicates sit next to each other
        per_creator[videos[video]] += _watch_seconds(evs)
    ranked = ((secs, creator) for creator, secs in per_creator.items() if secs > 0)
    best = heapq.nsmallest(k, ranked, key=lambda x: (-x[0], x[1]))
    return [(creator, secs) for secs, creator in best]

rng = random.Random(3030)
for _ in range(200):
    sample = rng.sample(watch_events, 300)
    k = rng.randint(1, 8)
    assert top_creators(sample, videos, k) == top_creators_brute(sample, videos, k)
print("stress test vs brute force: OK")              # prints: stress test vs brute force: OK
print(top_creators(watch_events, videos, 3))         # prints: [('c12', 163529), ('c11', 129188), ('c09', 69323)]
```

**Why:** Watch time only makes sense inside one viewing (one user on one video), so the events must be grouped by
that key before any arithmetic; summing per creator directly would pair a play of one user with a pause of
another. Sorting each group fixes the order of the unsorted log, and the two-state machine (open or closed) applies
rules 1 to 3 in one pass. Duplicates are harmless because a repeated play while open and a repeated pause while
closed are both ignored. The heap keeps the top k in O(C log k), which matters when there are millions of creators.

**Follow-ups the examiner may ask:**
- **SQL version**: `LEAD(ts) OVER (PARTITION BY user_id, video_id ORDER BY ts)` gives the next event; but double
  plays and orphan pauses make plain LEAD wrong, so first keep only state changes (a play after a close, a close after
  a play), then pair them.
- **Distributed**: shuffle by `(user_id, video_id)` to compute segments, then shuffle by `creator_id` to sum, then
  merge the per-worker top k lists. Taking a top k per worker before summing by creator would be wrong.
- **Real time**: a dashboard of top creators in the last hour: windowed sums per creator plus a heap; for huge
  cardinality, a Count-Min sketch with a heap of heavy hitters.
- Data quality: cap a segment at the video length or at an idle timeout, remove bots (thousands of views per hour),
  and report how many open segments were dropped (a crash rate metric).

**Complexity:** Brute force: O(C * n log n) (one scan per creator). Group, sort, sum: O(n log n) for the sorts, O(n) memory, plus O(C log k) for the top k.

**Common mistakes:** Summing play to pause across different users or videos. Sorting the whole log by time but not grouping. Counting an open segment up to the end of the day. Breaking ties in the wrong direction. Returning creators with 0 seconds.

**Learn more:** [algo-data-processing](https://mahsa-agz.github.io/ds-handbook/#algo-data-processing), [algo-heap](https://mahsa-agz.github.io/ds-handbook/#algo-heap), [algo-hashing](https://mahsa-agz.github.io/ds-handbook/#algo-hashing)

</details>

---
### Q3. Rapid drill: name the method (review)

*mixed, about 10 min*

Do not code. For each problem, say out loud in under 30 seconds: the method you would use and the
target time complexity. Write your answers as a short list, then open the solution.

1. A **sorted** list of prices and a budget: find two items whose total is exactly the budget.
2. The longest stretch of a string that contains at most 2 different characters.
3. Friendship pairs arrive one by one; after each pair, report how many separate friend groups exist.
4. A list of meetings `[start, end]`: the minimum number of rooms needed.
5. For each day's temperature, how many days until a warmer day.
6. The minimum number of coins (given denominations) to make an amount.
7. The smallest truck capacity that ships all packages, in their given order, within `D` days.
8. The shortest route from the start to the exit in a grid maze where every step costs the same.
9. All ways to split a string into pieces that are palindromes.
10. The 10 most frequent search queries in a log of 50 million rows.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Look for the signal words: **sorted**, **contiguous** / **at most**, **groups that merge**, **overlapping intervals**, **next greater**, **minimum number of ways / count**, **smallest value that works**, **shortest steps, equal cost**, **all ways**, **most frequent**.

</details>

<details><summary><b>Hint 2</b></summary>

Map of signals to methods: sorted + pair -> two pointers. Contiguous + at most k -> sliding window. Merging
groups over time -> union-find. Overlapping intervals at the same time -> sort + heap. Next greater element -> monotonic
stack. Optimal count over amounts -> DP. "Smallest value that works" and a yes/no check -> binary search on the
answer. Unweighted shortest path -> BFS. "All" combinations -> backtracking. Frequency + top k -> hash map + heap.

</details>

<details><summary><b>Solution</b></summary>

| # | Method | Target complexity |
|---|---|---|
| 1 | Two pointers from both ends (sorted input); a hash set also works in O(n) | O(n) time, O(1) space |
| 2 | Variable-size sliding window with a count map | O(n) |
| 3 | Union-find with path compression and union by size; groups = n minus successful unions | almost O(1) per pair |
| 4 | Sort by start, min-heap of end times (or a sweep over +1/-1 events) | O(n log n) |
| 5 | Monotonic stack of indexes with decreasing temperatures | O(n) |
| 6 | 1D DP over amounts: `dp[a] = 1 + min(dp[a - c])` | O(amount * coins) |
| 7 | Binary search on the capacity, greedy check "how many days with this capacity?" | O(n log(sum of weights)) |
| 8 | BFS from the start (all steps cost the same) | O(rows * cols) |
| 9 | Backtracking (cut a palindrome prefix, recurse); precompute a palindrome table to speed up checks | O(n * 2^n) worst case |
| 10 | Hash map counts, then a min-heap of size 10 (on many machines: count per partition by query, then merge) | O(n + U log 10), U = distinct queries |

**Why:** In the exam, recognizing the method in the first two minutes is most of the work. If one of these took you more than 30 seconds, re-read that section of the handbook tonight (only that one).

**Learn more:** [algo-recognize](https://mahsa-agz.github.io/ds-handbook/#algo-recognize), [cheat-algo-patterns](https://mahsa-agz.github.io/ds-handbook/#cheat-algo-patterns), [cheat-big-o](https://mahsa-agz.github.io/ds-handbook/#cheat-big-o)

</details>

---
## Checklist for the day before the exam

**Practice (at most 60 minutes):**
- Redo, from memory, the 3 problems in your mistake log that you missed most recently. No new problems today.
- Say the five-step script out loud once on any easy problem: clarify, examples, brute force, optimise, test.
- Skim the pattern cheat sheet and the Big O cheat sheet (10 minutes, no deep reading).

**Python toolkit you should type without thinking:**
- `collections`: `Counter`, `defaultdict(list)`, `deque` (`popleft`).
- `heapq`: `heappush`, `heappop`, `nsmallest`; max-heap by pushing `-x`; tuples `(priority, tie_breaker, item)`.
- `bisect.bisect_left` / `bisect_right`; `sorted(items, key=lambda x: (-x[1], x[0]))`.
- `functools.lru_cache` for memo, `sys.setrecursionlimit` or an explicit stack for deep recursion.
- `float("inf")`, `enumerate`, `zip`, list slicing, `"".join(...)`.

**Logistics:**
- Test the exam tool, camera, microphone and internet. Know how to run code in the shared editor.
- Prepare 3 questions for the examiner (team, data, how success is measured).
- Water, paper and pen on the desk.

**During the exam:**
- Clarify before coding. Say the brute force and its complexity even if you know the optimal method.
- Stuck for 5 minutes? Say what you are thinking and ask for a hint; a hint costs less than silence.
- Leave 5 minutes to test with your own small cases and edge cases (empty, one element, duplicates, very large).

**Tonight:** stop studying early, sleep 7 to 8 hours.

**Learn more:** [cheat-exam-day](https://mahsa-agz.github.io/ds-handbook/#cheat-exam-day), [cheat-algo-patterns](https://mahsa-agz.github.io/ds-handbook/#cheat-algo-patterns), [cheat-big-o](https://mahsa-agz.github.io/ds-handbook/#cheat-big-o), [algo-recognize](https://mahsa-agz.github.io/ds-handbook/#algo-recognize)

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [SQL](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_30/2_sql.ipynb)